# Table of Contents

- [Exploratory Data Analysis (EDA)](#eda)
- [Dataset Overview](#overview)
- [Missing Values](#missing)
- [Job Role Distribution](#roles)
- [Job Locations](#locations)
- [Salary Transparency Overview](#salary-overview)
- [Research Question Analysis](#rqa)
  - [Skills Demand](#skills)
    - [Top 15 Most Requested Skills](#top15)
    - [Skill Count Distribution by Role (Box Plot)](#boxplot)
    - [Top 10 Skills by Role (Heatmap)](#heatmap-role)
  - [Regional & Industry Differences](#regional)
    - [Skill Requirements by Canton](#canton-skills)
    - [Skill Requirements by Industry (Heatmap)](#heatmap-industry)
    - [Role Distribution by Region](#region-role)
  - [Seniority Differences](#seniority)
    - [Top 10 Skills by Seniority (Heatmap)](#heatmap-seniority)
  - [Skill Co-occurrence](#cooccur)
  - [Workload Distribution](#workload)
  - [Macro Labor Market Context](#macro)
  - [Salary Transparency Patterns](#salary-patterns)
- [Summary of Findings](#summary)
- [Data Limitations](#limitations)


# Exploratory Data Analysis (EDA)

This section explores the merged dataset containing Swiss job postings and macro-level vacancy statistics from the Swiss Federal Statistical Office (BFS). The objective is to understand the dataset structure, identify potential data quality issues, and explore key variables before addressing the research questions.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import ast

BG     = "#F8F9FB"
BLUE   = "#2C6E9E"
TEAL   = "#2CA4A0"
ORANGE = "#E07B39"
GREEN  = "#5BAD72"
PURPLE = "#7B5EA7"
COLORS = [BLUE, TEAL, ORANGE, GREEN, PURPLE, "#C94F6D", "#A0853C"]

sns.set(style="whitegrid")

## Load the Dataset

The dataset contains merged micro-level job postings and macro-level BFS vacancy statistics.

In [ ]:
df = pd.read_csv("../data/processed/jobs_merged_final.csv")
df.head()

## Dataset Overview

In [ ]:
print("Dataset shape:", df.shape)
df.info()

In [ ]:
df.describe()

The descriptive statistics summarize numerical variables including skill counts and vacancy indicators.

## Missing Values

Identifying missing data helps understand dataset limitations and guides analysis decisions.

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values

`salary_min`, `salary_max`, and `seniority` contain the most missing values. Salary is rarely disclosed in Swiss job ads. Seniority is not always stated explicitly in job titles. Regional nulls are due to unresolvable location strings in the raw data.

## Job Role Distribution

In [ ]:
role_counts = df["role"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4), facecolor=BG)
ax.set_facecolor(BG)
bars = ax.barh(role_counts.index[::-1], role_counts.values[::-1],
               color=COLORS[:len(role_counts)][::-1], edgecolor="none", height=0.6)
for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width())}", va='center', fontsize=9, color="#333")
ax.set_xlabel("Number of Postings")
ax.set_title("Job Role Distribution", fontsize=13, fontweight='bold', pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

Data Engineer is the most frequent role (448), followed by Data Scientist (122) and Data Analyst (95). ML Engineer and AI Engineer are more niche roles.

## Job Locations

In [ ]:
loc_counts = df["canton"].value_counts()
cmap = plt.cm.Blues
norm_vals = (loc_counts.values - loc_counts.values.min()) / (loc_counts.values.max() - loc_counts.values.min())
col = [cmap(0.3 + 0.65 * v) for v in norm_vals]

fig, ax = plt.subplots(figsize=(8, 6), facecolor=BG)
ax.set_facecolor(BG)
bars = ax.barh(loc_counts.index[::-1], loc_counts.values[::-1], color=col[::-1], edgecolor="none", height=0.7)
for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width())}", va='center', fontsize=8, color="#333")
ax.set_xlabel("Postings")
ax.set_title("Job Postings by Canton", fontsize=13, fontweight='bold', pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

ZH (242), GE (87), VD (76), BE (59) dominate — the top 4 cantons account for 65% of all postings, reflecting major Swiss economic and technology hubs.

## Salary Transparency Overview

In [ ]:
salary_rate = df["salary_available"].mean()
print(f"Salary transparency rate: {round(salary_rate * 100, 2)} %")
print(f"Postings with salary: {df['salary_available'].sum()} / {len(df)}")

Salary information is provided in only **1.2% of job postings**, indicating extremely low salary transparency. Most employers omit compensation details and disclose them later in the recruitment process.

---
# Research Question Analysis

The following sections examine the three research questions from the feasibility study: skill demand, regional and industry differences, and salary transparency patterns.

## Skills Demand

The first research question examines which skills are most frequently requested in Swiss data job postings.

### Helper: Parse Skills Column

In [ ]:
def parse_skills(x):
    try:
        return [s.strip().lower() for s in ast.literal_eval(x)]
    except:
        return []

df["skills_list"] = df["skills"].apply(parse_skills)
all_skills = [s for sl in df["skills_list"] for s in sl]
print(f"Total skill mentions: {len(all_skills)}")

### Top 15 Most Requested Skills

In [ ]:
top15 = pd.DataFrame(Counter(all_skills).most_common(15), columns=["skill", "count"])

cmap2 = plt.colormaps["YlGnBu"]
cols = [cmap2(0.35 + 0.6 * (i / 14)) for i in range(15)]

fig, ax = plt.subplots(figsize=(8, 5), facecolor=BG)
ax.set_facecolor(BG)
bars = ax.barh(top15["skill"][::-1], top15["count"][::-1],
               color=cols[::-1], edgecolor="none", height=0.65)
for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width())}", va='center', fontsize=8.5, color="#333")
ax.set_xlabel("Postings Mentioning Skill")
ax.set_title("Top 15 Most Requested Skills", fontsize=13, fontweight='bold', pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

Python leads by a wide margin (165 mentions), followed by SQL (99), Azure (96), CI/CD (88) and Machine Learning (86). The mix of programming languages, cloud platforms, and ML frameworks reflects the breadth of modern data roles.

### Skill Count Distribution by Role

In [ ]:
role_order = df.groupby("role")["skill_count"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=BG)
ax.set_facecolor(BG)
data_box = [df[df["role"] == r]["skill_count"].values for r in role_order]
bp = ax.boxplot(data_box, vert=False, patch_artist=True,
                medianprops=dict(color="white", linewidth=2),
                whiskerprops=dict(color="#888"),
                capprops=dict(color="#888"),
                flierprops=dict(marker='o', color="#888", alpha=0.4, markersize=4))
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
ax.set_yticks(range(1, len(role_order) + 1))
ax.set_yticklabels(role_order)
ax.set_xlabel("Skill Count per Posting")
ax.set_title("Skill Count Distribution by Role", fontsize=13, fontweight='bold', pad=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

ML Engineer shows the widest spread — some postings list 15+ skills. Data Engineer clusters near zero because many non-DS postings were scraped alongside genuine data engineering roles.

### Top 10 Skills by Role (Heatmap)

In [ ]:
top10 = [s for s, _ in Counter(all_skills).most_common(10)]
roles = df["role"].unique()

rows_r = []
for role in roles:
    sub = df[df["role"] == role]
    all_s = [s for sl in sub["skills_list"] for s in sl]
    total = len(sub)
    rows_r.append({sk: all_s.count(sk) / max(total, 1) * 100 for sk in top10})

hm_role = pd.DataFrame(rows_r, index=roles)[top10]

fig, ax = plt.subplots(figsize=(10, 4), facecolor=BG)
sns.heatmap(hm_role, ax=ax, cmap="PuBuGn", annot=True, fmt=".0f",
            linewidths=0.5, linecolor="#eee",
            cbar_kws={"label": "% of postings in role"})
ax.set_title("Top 10 Skills by Role (% of postings)", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Python dominates across all roles. Machine learning is a signature skill for ML Engineer and Data Scientist. CI/CD and Kubernetes are most prominent in Data Engineer postings, reflecting infrastructure-heavy requirements.

## Regional & Industry Differences

This section examines whether job requirements differ across regions and industries in Switzerland.

### Skill Requirements by Canton

In [ ]:
skills_canton = (
    df[df["canton"].notna()]
    .groupby("canton")["skill_count"]
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5), facecolor=BG)
ax.set_facecolor(BG)
bars = ax.barh(skills_canton.index[::-1], skills_canton.values[::-1],
               color=BLUE, edgecolor="none", height=0.65, alpha=0.85)
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{bar.get_width():.1f}", va='center', fontsize=8, color="#333")
ax.set_xlabel("Average Skill Count")
ax.set_title("Average Skill Requirements by Canton", fontsize=13, fontweight='bold', pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

ZG and SZ lead but have very few postings. ZH and GE show consistently high average skill requirements, reflecting the concentration of tech companies in these cantons. Interpret with caution — cantons with fewer than 10 postings have high variance.

### Top 10 Skills by Industry (Heatmap)

In [ ]:
top_inds = df["industry"].value_counts()
top_inds = top_inds[top_inds >= 20].index.tolist()

rows_i = []
for ind in top_inds:
    sub = df[df["industry"] == ind]
    all_s = [s for sl in sub["skills_list"] for s in sl]
    total = len(sub)
    rows_i.append({sk: all_s.count(sk) / max(total, 1) * 100 for sk in top10})

hm_ind = pd.DataFrame(rows_i, index=[i.split(" ", 1)[-1][:30] for i in top_inds])[top10]

fig, ax = plt.subplots(figsize=(11, 5), facecolor=BG)
sns.heatmap(hm_ind, ax=ax, cmap="YlOrRd", annot=True, fmt=".0f",
            linewidths=0.5, linecolor="#eee",
            cbar_kws={"label": "% of postings in industry"})
ax.set_title("Top 10 Skills by Industry (% of postings)", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

IT services heavily favors CI/CD and Azure. Finance leans toward SQL and Python. Education shows machine learning and R prominently, reflecting the academic research context of those postings.

### Role Distribution by Region

In [ ]:
top_regions = df["region"].value_counts().head(6).index
rr = (df[df["region"].isin(top_regions)]
      .groupby(["region", "role"])
      .size()
      .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(BG)
rr.plot(kind='bar', ax=ax, color=COLORS[:len(rr.columns)], edgecolor="white", linewidth=0.5, width=0.75)
ax.set_xlabel("")
ax.set_ylabel("Number of Postings")
ax.set_title("Role Distribution by Region", fontsize=13, fontweight='bold', pad=12)
ax.legend(title="Role", bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

Zurich and Lake Geneva lead across all roles. Data Engineer postings are disproportionately concentrated in Zurich, while Lake Geneva shows a higher relative share of Data Scientist roles, likely driven by EPFL and research institutions.

## Seniority Differences

This section examines how skill requirements vary across seniority levels.

### Top 10 Skills by Seniority Level (Heatmap)

In [ ]:
sen_order = ["intern", "junior", "mid", "senior", "lead"]
rows_s = []
for sen in sen_order:
    sub = df[df["seniority"] == sen]
    all_s = [s for sl in sub["skills_list"] for s in sl]
    total = len(sub)
    rows_s.append({sk: all_s.count(sk) / max(total, 1) * 100 for sk in top10})

hm_sen = pd.DataFrame(rows_s, index=sen_order)[top10]

fig, ax = plt.subplots(figsize=(10, 4), facecolor=BG)
sns.heatmap(hm_sen, ax=ax, cmap="Blues", annot=True, fmt=".0f",
            linewidths=0.5, linecolor="#eee",
            cbar_kws={"label": "% of postings in seniority level"})
ax.set_title("Top 10 Skills by Seniority Level (% of postings)", fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Senior and lead roles show higher Python and cloud skill requirements. Interns show higher R and machine learning mentions, reflecting academic or research contexts. CI/CD and Kubernetes are most prominent at senior and lead levels.

## Skill Co-occurrence

To understand which skills are requested together, we analyze skills that co-occur most frequently with Python — the dominant language in the dataset.

In [ ]:
python_jobs = df[df["skills_list"].apply(lambda x: "python" in x)]
co_skills = [s for sl in python_jobs["skills_list"] for s in sl if s != "python"]
co_top = pd.DataFrame(Counter(co_skills).most_common(10), columns=["skill", "count"])

fig, ax = plt.subplots(figsize=(8, 4), facecolor=BG)
ax.set_facecolor(BG)
bars = ax.barh(co_top["skill"][::-1], co_top["count"][::-1],
               color=TEAL, edgecolor="none", height=0.6)
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{int(bar.get_width())}", va='center', fontsize=9, color="#333")
ax.set_xlabel("Co-occurrences with Python")
ax.set_title("Skills Most Frequently Paired with Python", fontsize=13, fontweight='bold', pad=12)
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

SQL, machine learning, and CI/CD are the most common skills co-occurring with Python — defining the core Swiss data stack. The combination of Python + SQL + cloud tools reflects a full-stack data engineering profile.

## Workload Distribution by Role

In [ ]:
def workload_cat(row):
    if pd.isna(row["workload_min"]):
        return "Unknown"
    if row["workload_max"] <= 60:
        return "Part-time (≤60%)"
    if row["workload_min"] >= 90:
        return "Full-time (≥90%)"
    return "Flexible (61-89%)"

df["workload_cat"] = df.apply(workload_cat, axis=1)
wl = df.groupby(["role", "workload_cat"]).size().unstack(fill_value=0)
wl_pct = wl.div(wl.sum(axis=1), axis=0) * 100

wl_colors = {
    "Full-time (≥90%)":  BLUE,
    "Flexible (61-89%)": TEAL,
    "Part-time (≤60%)":  ORANGE,
    "Unknown":           "#ccc"
}

fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=BG)
ax.set_facecolor(BG)
bottom = np.zeros(len(wl_pct))
for cat in ["Full-time (≥90%)", "Flexible (61-89%)", "Part-time (≤60%)", "Unknown"]:
    if cat in wl_pct.columns:
        vals = wl_pct[cat].values
        ax.bar(wl_pct.index, vals, bottom=bottom, label=cat,
               color=wl_colors[cat], edgecolor="white", linewidth=0.5)
        bottom += vals
ax.set_ylabel("% of Postings")
ax.set_title("Workload Distribution by Role", fontsize=13, fontweight='bold', pad=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

Full-time arrangements dominate across all roles. AI Engineer shows the highest share of flexible workload options. Part-time roles are relatively rare in data-related positions.

## Macro Labor Market Context

To validate the representativeness of the scraped data, job posting counts are compared with official BFS vacancy statistics at the regional level.

In [ ]:
rc = (df[df["region"].notna()]
      .groupby("region")
      .agg(job_postings=("job_id", "count"),
           macro_vacancies=("macro_vacancies_region", "mean"))
      .reset_index()
      .sort_values("job_postings", ascending=True))

fig, ax1 = plt.subplots(figsize=(9, 5), facecolor=BG)
ax1.set_facecolor(BG)
ax2 = ax1.twinx()
ax1.barh(rc["region"], rc["job_postings"], color=BLUE, alpha=0.8, label="Job Postings", height=0.5)
ax2.plot(rc["macro_vacancies"], rc["region"], "o-", color=ORANGE,
         linewidth=2, markersize=7, label="BFS Vacancies")
ax1.set_xlabel("Job Postings", color=BLUE)
ax2.set_xlabel("BFS Macro Vacancies", color=ORANGE)
ax1.set_title("Scraped Postings vs BFS Vacancies by Region", fontsize=13, fontweight='bold', pad=12)
ax1.spines[['top']].set_visible(False)
ax2.spines[['top']].set_visible(False)
h1 = mpatches.Patch(color=BLUE, label='Job Postings')
h2 = plt.Line2D([0], [0], color=ORANGE, marker='o', label='BFS Vacancies')
ax1.legend(handles=[h1, h2], loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

The distribution of scraped job postings broadly aligns with official BFS vacancy data — Zurich leads both metrics, supporting the representativeness of the dataset. Some divergence is expected given the scraped data focuses exclusively on data-related roles.

## Salary Transparency Patterns

The final research question examines whether salary transparency varies by canton and role.

In [ ]:
sal_c = (df[df["canton"].notna()]
         .groupby("canton")["salary_available"]
         .mean()
         .sort_values(ascending=False))
sal_c = sal_c[sal_c > 0]

sal_r = (df.groupby("role")["salary_available"]
         .mean()
         .sort_values(ascending=False))
sal_r = sal_r[sal_r > 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5), facecolor=BG)
for ax in [ax1, ax2]:
    ax.set_facecolor(BG)

ax1.barh(sal_c.index[::-1], sal_c.values[::-1] * 100, color=PURPLE, edgecolor="none", height=0.5)
ax1.set_xlabel("% with Salary")
ax1.set_title("Salary Transparency by Canton", fontsize=11, fontweight='bold')
ax1.spines[['top', 'right', 'left']].set_visible(False)
ax1.tick_params(left=False)

ax2.barh(sal_r.index[::-1], sal_r.values[::-1] * 100, color=GREEN, edgecolor="none", height=0.4)
ax2.set_xlabel("% with Salary")
ax2.set_title("Salary Transparency by Role", fontsize=11, fontweight='bold')
ax2.spines[['top', 'right', 'left']].set_visible(False)
ax2.tick_params(left=False)

plt.tight_layout()
plt.show()

Only BE, ZH, and GE show any salary disclosure. Only Data Scientist and Data Engineer roles include salary information. Overall transparency remains extremely limited — consistent with Swiss labor market norms where compensation is typically discussed during the interview process.

## Summary of Findings

The analysis reveals several key patterns in the Swiss data job market:

**Role demand:** Data Engineer is the most advertised role, followed by Data Scientist and Data Analyst. ML Engineer and AI Engineer are niche but growing.

**Skills:** Python, SQL, and Azure are the most requested skills across all roles. Heatmaps reveal clear skill signatures per role — CI/CD and Kubernetes dominate Data Engineering, while machine learning and R characterize Data Science and academic roles.

**Regional concentration:** Job postings are concentrated in ZH, GE, VD, and BE. The distribution broadly aligns with BFS macro vacancy data, supporting dataset representativeness.

**Industry differences:** IT services and Finance show the highest skill requirements. ICT sector postings are heavily cloud and DevOps oriented.

**Seniority:** Senior and lead roles require more diverse technical skills. Interns and juniors show more focused, often R-based profiles reflecting academic settings.

**Salary transparency:** Extremely low at 1.2% overall. Only BE, ZH, and GE show any disclosure. This pattern itself is a key finding — salary transparency is not yet standard practice in Swiss job advertising.

## Data Limitations

- **Location nulls (13.3%):** Some postings could not be mapped to a canton due to ambiguous location strings (company names, region labels). Regional comparisons may be slightly affected.
- **Salary sparsity (98.8% null):** Only 9 of 743 postings include salary data. Salary analyses should be interpreted as transparency indicators, not market benchmarks.
- **Seniority nulls (53%):** Seniority classification relies on job title keywords; many titles do not indicate level explicitly.
- **Snapshot data:** All postings were collected within a short time window in early 2026. Findings reflect a market snapshot rather than longitudinal trends.
- **Scraper noise:** ~38% of postings have zero skills extracted, partly because non-DS postings were included in the scrape. Skill analyses are most meaningful for postings with skill_count > 0.